# pyHidroWeb Tutorial

**pyHidroWeb** is a Python library for downloading Brazilian hydrological data from the [HidroWeb portal](https://www.snirh.gov.br/hidroweb/) of the National Water Agency (ANA).

This tutorial covers:

1. [Installation](#1.-Installation)
2. [Finding Stations](#2.-Finding-Stations)
3. [Downloading Data for a Single Station](#3.-Downloading-Data-for-a-Single-Station)
4. [Output Formats: DataFrame and xarray](#4.-Output-Formats:-DataFrame-and-xarray)
5. [Filtering by Date and Consistency Level](#5.-Filtering-by-Date-and-Consistency-Level)
6. [Batch Downloads](#6.-Batch-Downloads)
7. [Error Handling](#7.-Error-Handling)
8. [Data Analysis Example](#8.-Data-Analysis-Example)

## 1. Installation

Install the package with all optional dependencies:

```bash
# Core package (pandas + requests only)
pip install pyhydroweb

# With xarray support (needed for output_format=1)
pip install pyhydroweb[xarray]

# With geopandas support (needed for shapefile / polygon station selection)
pip install pyhydroweb[geopandas]

# Everything
pip install pyhydroweb[all]
```

Or, from the repository root:

```bash
pip install -e ".[all]"
```

## 2. Finding Stations

Before downloading data you need a station code. pyHidroWeb can query the official SNIRH station inventory and filter it by location, distance, or polygon area.

Station codes can also be looked up directly on the [HidroWeb portal](https://www.snirh.gov.br/hidroweb/serieshistoricas).

**Station types:**
- `FLU` — Fluviometric (measures river discharge / water level)
- `PLU` — Pluviometric (measures rainfall)

In [ ]:
from pyhydroweb import (
    get_station_metadata,
    get_nearby_stations,
    get_stations_in_bounds,
    get_stations_in_polygon,
    get_all_stations,
    clear_metadata_cache,
)

### 2.1 List all stations

`get_all_stations()` returns the full inventory. The first call downloads the inventory from SNIRH and caches it at `~/.pyhydroweb/metadata/stations.csv` for 30 days.

In [ ]:
# All stations (both types)
all_stations = get_all_stations()
print(f"Total stations in inventory: {len(all_stations)}")
all_stations.head()

In [ ]:
# Fluviometric stations only
flu_stations = get_all_stations(station_type="FLU")
print(f"Fluviometric stations: {len(flu_stations)}")

# Pluviometric stations only
plu_stations = get_all_stations(station_type="PLU")
print(f"Pluviometric stations: {len(plu_stations)}")

### 2.2 Stations within geographic bounds

In [ ]:
# Find all fluviometric stations in a bounding box
stations_bbox = get_stations_in_bounds(
    min_lat=-16, max_lat=-15,
    min_lon=-46, max_lon=-45,
    station_type="FLU",
)
print(f"Found {len(stations_bbox)} stations in the bounding box")
stations_bbox[["Codigo", "Nome", "Latitude", "Longitude", "TipoEstacao"]]

### 2.3 Stations near a point

In [ ]:
# Find all stations within 100 km of a coordinate
nearby = get_nearby_stations(
    latitude=-15.5,
    longitude=-45.5,
    radius_km=100,
    station_type="FLU",
)
print(f"Found {len(nearby)} stations within 100 km")
nearby[["Codigo", "Nome", "distance_km", "Latitude", "Longitude"]].head(10)

### 2.4 Stations within a polygon (requires geopandas)

Polygon coordinates are given as `(latitude, longitude)` pairs.

In [ ]:
try:
    polygon = [
        (-16.0, -46.0),
        (-16.0, -45.0),
        (-15.0, -45.0),
        (-15.0, -46.0),
    ]
    stations_poly = get_stations_in_polygon(polygon, station_type="FLU")
    print(f"Found {len(stations_poly)} stations inside polygon")
    stations_poly[["Codigo", "Nome", "Latitude", "Longitude"]]
except Exception as e:
    print(f"Skipped: {e}")

### 2.5 Metadata for a specific station

In [ ]:
# Station 34879500 — Rio Parnaíba at Luzilândia
meta = get_station_metadata("34879500")
if meta:
    print(f"Name      : {meta['Nome']}")
    print(f"Type      : {meta['TipoEstacao']}")
    print(f"Latitude  : {meta['Latitude']}")
    print(f"Longitude : {meta['Longitude']}")
    print(f"Altitude  : {meta.get('Altitude')} m")

### 2.6 Refreshing the metadata cache

The inventory is cached locally for 30 days. To force a fresh download:

In [ ]:
from pyhydroweb import get_cached_inventory

# Force a fresh download regardless of cache age
# inventory = get_cached_inventory(force_refresh=True)

# Or clear the entire cache directory
# clear_metadata_cache()

## 3. Downloading Data for a Single Station

`download_hidroweb_data()` retrieves the full historical series for one station from the ANA SOAP API.

| Parameter | Values | Default |
|---|---|---|
| `data_type` | `2` = Rainfall, `3` = Flow | `3` |
| `consistency_level` | `0` = Both, `1` = Raw only, `2` = Consolidated only | `0` |
| `output_format` | `0` = pandas DataFrame, `1` = xarray Dataset | `1` |

In [ ]:
from pyhydroweb import download_hidroweb_data

# Download the full flow series as a pandas DataFrame
df = download_hidroweb_data(
    station=34879500,
    data_type=3,          # 3 = Flow (Vazão)
    output_format=0,      # 0 = pandas DataFrame
)
print(df.shape)
df.head()

In [ ]:
# Download rainfall data
df_rain = download_hidroweb_data(
    station=34879500,
    data_type=2,          # 2 = Rainfall (Chuva)
    output_format=0,
)
print(df_rain.shape)
df_rain.head()

## 4. Output Formats: DataFrame and xarray

The default output (`output_format=1`) is an **xarray Dataset**, which is convenient for labelled multidimensional operations. Use `output_format=0` for a plain **pandas DataFrame**.

In [ ]:
# xarray Dataset (default)
ds = download_hidroweb_data(station=34879500, data_type=3, output_format=1)
ds

In [ ]:
# Station metadata is stored in .attrs on both DataFrame and Dataset
print("Station name :", ds.attrs.get("station_name"))
print("Latitude     :", ds.attrs.get("latitude"))
print("Longitude    :", ds.attrs.get("longitude"))
print("Altitude     :", ds.attrs.get("altitude"), "m")

In [ ]:
# Same metadata is available on the DataFrame
df = download_hidroweb_data(station=34879500, data_type=3, output_format=0)
print("Station name :", df.attrs.get("station_name"))
print("Latitude     :", df.attrs.get("latitude"))

## 5. Filtering by Date and Consistency Level

Use `start_date` and `end_date` (format `YYYY-MM-DD`) to limit the download to a specific period.

**Consistency levels:**
- `0` — both raw and consolidated records (default)
- `1` — raw data only (real-time, may have gaps)
- `2` — consolidated data only (quality-controlled)

In [ ]:
# Download a specific year of consolidated flow data
df_2020 = download_hidroweb_data(
    station=34879500,
    start_date="2020-01-01",
    end_date="2020-12-31",
    data_type=3,
    consistency_level=2,   # 2 = consolidated only
    output_format=0,
)
print(f"Records: {len(df_2020)}")
df_2020.head()

In [ ]:
# Inspect unique consistency levels in the result
print(df_2020["flow_rate_consistency_level"].value_counts())
# 1 = Raw, 2 = Consolidated

## 6. Batch Downloads

### 6.1 Download from a list of stations

`download_from_list()` fetches CSVs directly from the ANA GitHub repositories and saves them to a local directory. It supports both **automatic** (telemetry) and **conventional** stations.

| Parameter | Description |
|---|---|
| `stations_list` | List of station code strings |
| `date_limits` | `["YYYY-MM-DD", "YYYY-MM-DD"]` to filter dates |
| `path_dir` | Directory where CSV files are saved |
| `automatic_stations` | `True` = automatic/telemetry, `False` = conventional |
| `hidrologic_variable` | `"fluviometricas"` or `"pluviometricas"` |
| `data_type` | `"vazoes"` (flow) or `"cotas"` (water level) — conventional only |

In [ ]:
from pyhydroweb import download_from_list
import os

stations = ["34879500", "34880000"]
output_dir = "./hydro_data"

download_from_list(
    stations_list=stations,
    date_limits=["2022-01-01", "2022-12-31"],
    path_dir=output_dir,
    automatic_stations=True,
    hidrologic_variable="fluviometricas",
)

print("Files saved:")
for f in os.listdir(output_dir):
    print(" ", f)

For **conventional stations**, set `automatic_stations=False` and specify `data_type`:

In [ ]:
# Conventional stations — flow data
# download_from_list(
#     stations_list=["10042000"],
#     path_dir="./conventional_data",
#     automatic_stations=False,
#     data_type="vazoes",
#     hidrologic_variable="fluviometricas",
# )

### 6.2 Download for all stations within a shapefile (requires geopandas)

`download_from_shape()` reads a shapefile, finds all HidroWeb stations within its boundary, and downloads their data.

In [ ]:
from pyhydroweb import download_from_shape

# download_from_shape(
#     shape="study_area.shp",
#     date_limits=["2020-01-01", "2020-12-31"],
#     path_dir="./shape_data",
#     automatic_stations=True,
#     hidrologic_variable="fluviometricas",
# )

## 7. Error Handling

All exceptions inherit from `PyHidroWebError`, so you can catch the base class or handle specific cases:

In [ ]:
from pyhydroweb import PyHidroWebError
from pyhydroweb.exceptions import (
    InvalidDataTypeError,
    InvalidDateFormatError,
    InvalidDateRangeError,
    InvalidStationCodeError,
    DownloadError,
    MissingDependencyError,
)

In [ ]:
# Invalid data type
try:
    download_hidroweb_data(34879500, data_type=5)
except InvalidDataTypeError as e:
    print(f"InvalidDataTypeError: {e}")

# Invalid date format
try:
    download_hidroweb_data(34879500, start_date="01/01/2020")
except InvalidDateFormatError as e:
    print(f"InvalidDateFormatError: {e}")

# Start date after end date
try:
    download_hidroweb_data(34879500, start_date="2020-12-31", end_date="2020-01-01")
except InvalidDateRangeError as e:
    print(f"InvalidDateRangeError: {e}")

# Invalid station code
try:
    download_hidroweb_data("ABC123")
except InvalidStationCodeError as e:
    print(f"InvalidStationCodeError: {e}")

## 8. Data Analysis Example

Here we download the full flow series for station 34879500 (Rio Parnaíba at Luzilândia) and explore it with xarray and matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ds = download_hidroweb_data(station=34879500, data_type=3, output_format=1)
ds

### 9.1 Seasonal statistics

Group the daily series by calendar month and compute the 25th, 50th, and 75th percentiles.

In [ ]:
quantiles = ds.flow_rate.groupby("time.month").quantile([0.25, 0.5, 0.75])
quantiles.attrs = ds.flow_rate.attrs
quantiles

### 9.2 Full time series + seasonal spread

In [ ]:
months = "J F M A M J J A S O N D".split()

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
fig.subplots_adjust(hspace=0.5)

# Top: full daily time series
ds.flow_rate.plot(ax=ax[0])
ax[0].set_title(f"{ds.attrs.get('station_name', 'Station')} — daily flow")

# Bottom: seasonal IQR + median
ax[1].plot(
    [quantiles.month] * 2,
    quantiles.sel(quantile=[0.25, 0.75]).values.T,
    marker="_",
    color="0.4",
    label="IQR (25–75%)",
)
quantiles.sel(quantile=0.5).plot(ax=ax[1], marker="o", label="Median")
ax[1].set(xticks=np.arange(len(months)) + 1, xticklabels=months, title="Seasonal cycle")
ax[1].legend()

plt.show()

### 9.3 Identify anomalous years

Find years where the April mean flow was above the 75th percentile (wet years) or below the 25th percentile (dry years).

In [ ]:
april_annual = (
    ds.flow_rate.where(ds.time.dt.month == 4, drop=True)
    .groupby("time.year")
    .mean()
)

q75 = float(quantiles.sel(month=4, quantile=0.75))
q25 = float(quantiles.sel(month=4, quantile=0.25))

wet_years = april_annual.year.where(april_annual > q75, drop=True).values.astype(int)
dry_years = april_annual.year.where(april_annual < q25, drop=True).values.astype(int)

print(f"Wet Aprils  (> 75th pct): {wet_years}")
print(f"Dry Aprils  (< 25th pct): {dry_years}")

### 9.4 Compare a specific year against the climatology

In [ ]:
year = 1998  # El Niño year — change to any year of interest

dsy = ds.where(ds.time.dt.year == year, drop=True)

months = "J F M A M J J A S O N D".split()
fig, ax = plt.subplots(2, 1, figsize=(10, 6))
fig.subplots_adjust(hspace=0.5)

# Top: full series with selected year highlighted
ds.flow_rate.plot(ax=ax[0], alpha=0.5, label="All years")
dsy.flow_rate.plot(ax=ax[0], color="tab:red", label=str(year))
ax[0].set_title(f"{ds.attrs.get('station_name', 'Station')} — {year} highlighted")
ax[0].legend()

# Bottom: seasonal cycle + selected year's monthly median
ax[1].plot(
    [quantiles.month] * 2,
    quantiles.sel(quantile=[0.25, 0.75]).values.T,
    marker="_",
    color="0.4",
)
quantiles.sel(quantile=0.5).plot(ax=ax[1], marker="o", label="Median (all years)")
dsy.groupby("time.month").median().flow_rate.plot(
    x="month", ax=ax[1], color="tab:red", marker="o", label=str(year)
)
ax[1].set(xticks=np.arange(len(months)) + 1, xticklabels=months, title="Seasonal comparison")
ax[1].legend()

plt.show()

## Appendix: Logging

Enable verbose logging to monitor downloads and debug issues:

In [ ]:
import logging
from pyhydroweb import setup_logging

# Set to DEBUG for maximum verbosity, INFO for normal use
setup_logging(level=logging.INFO)